In [64]:
from padel import *

print(hold_star(0.6))
print(set_prob(0.62, 0.58))
print(match_prob(0.62, 0.58))

0.7298491391999999
0.632588187725585
0.6942205796796874


In [59]:
import random
p = 0.6
n = 100000
wins = sum(random.random() < p for _ in range(n))
print(wins / n)

0.60167


In [61]:
def sim_game_golden(p):
    a, b = 0, 0
    while True:
        if a==3 and b==3:
            return random.random() < p
        if random.random() < p:
            a += 1
        else:
            b += 1
        if a==4:
            return True
        if b==4:
            return False

In [62]:
for _ in range(10):
    print(sim_game_golden(0.6))

True
True
True
False
True
True
True
False
False
True


In [170]:
n = 1000
holds = sum(sim_game_golden(0.6) for _ in range(n))
print(holds / n)
print(hold_golden(0.6))

est = holds / n
se = (est * (1 - est) / n) ** 0.5

print("SE:", se)
print("difference:", abs(est - hold_golden(0.6)))
print("agrees:", abs(est - hold_golden(0.6)) < 2 * se)

0.729
0.710208
SE: 0.014055568291606
difference: 0.01879200000000003
agrees: True


In [173]:
def sim_game_deuce(p):
    a, b = 0, 0
    while True:
        if random.random() < p:
            a += 1
        else:
            b += 1
        if a>=4 and a-b>=2:
            return True
        if b>=4 and b-a>=2:
            return False

In [277]:
n = 100000
holds = sum(sim_game_deuce(0.6) for _ in range(n))
print(holds / n)
print(hold_deuce(0.6))

est = holds / n
se = (est * (1 - est) / n) ** 0.5

print("SE:", se)
print("difference:", abs(est - hold_deuce(0.6)))
print("agrees:", abs(est - hold_deuce(0.6)) < 2 * se)

0.73353
0.7357292307692308
SE: 0.0013980834706840647
difference: 0.0021992307692307866
agrees: True


In [328]:
def sim_game_star(p):
    a, b = 0, 0
    deuces = 0
    while True:
        if random.random() < p:
            a += 1
        else:
            b += 1
        if a==b and a>=3:
            deuces+=1
        if a>=4 and a-b>=2:
            return True
        if b>=4 and b-a>=2:
            return False
        if deuces == 3:
            return random.random() < p

In [331]:
n = 100000
holds = sum(sim_game_star(0.6) for _ in range(n))
print(holds / n)
print(hold_star(0.6))

est = holds / n
se = (est * (1 - est) / n) ** 0.5

print("SE:", se)
print("difference:", abs(est - hold_star(0.6)))
print("agrees:", abs(est - hold_star(0.6)) < 2 * se)

0.72756
0.7298491391999999
SE: 0.0014078936266636057
difference: 0.0022891391999999566
agrees: True


In [ ]:
def sim_tiebreak(p_a, p_b, first="A"):
    a, b = 0, 0
    while True:
        n = a + b
        if ((n+1) // 2) % 2 == 0:
            server = first
        else:
            server = "B" if first == "A" else "A"

        x = p_a if server == "A" else 1 - p_b

        if random.random() < x:
            a += 1
        else:
            b += 1
        
        if a >= 7 and a - b >= 2:
            return True
        if b >= 7 and b - a >= 2:
            return False



In [334]:
def sim_set(p_a, p_b, game=sim_game_star, server="A"):
    ga, gb = 0, 0
    while True:
        if server == "A":
            a_won_game = game(p_a)
        else:
            a_won_game = not game(p_b)
        
        if a_won_game:
            ga += 1
        else:
            gb += 1
        
        server = "B" if server == "A" else "A"

        if(ga>=6 and ga-gb>=2) or ga==7:
            return True, server
        if(gb>=6 and gb-ga>=2) or gb==7:
            return False, server
        
        if ga==6 and gb==6:
            a_won = sim_tiebreak(p_a, p_b, first=server)
            next_server = "B" if server == "A" else "A"
            return a_won, next_server


def sim_match(p_a, p_b, game=sim_game_star, server="A"):
    sa, sb = 0, 0
    while sa < 2 and sb < 2:
        a_won, server = sim_set(p_a, p_b, game, server)
        if a_won:
            sa += 1
        else:
            sb += 1
    return sa == 2

In [339]:
def check(sim_fn, exact, n=20_000):
    """Run sim_fn n times, compare the win rate with the exact answer."""
    wins = sum(sim_fn() for _ in range(n))
    est = wins / n
    se = (est * (1 - est) / n) ** 0.5
    ok = abs(est - exact) < 2 * se
    print(f"Sim: {est:.4f}   Exact: {exact:.4f}   SE: {se:.4f}   {'OK' if ok else 'CHECK'}")

In [ ]:
check(lambda: sim_match(0.62, 0.58), match_prob(0.62, 0.58))

Sim: 0.6883   Exact: 0.6942   SE: 0.0033   OK


In [346]:
rules = [("Golden",    sim_game_golden, hold_golden),
("Deuce", sim_game_deuce,  hold_deuce),
("Star",      sim_game_star,   hold_star),
]

for name, sim_game, hold in rules:
    for p_a, p_b in [(0.6, 0.6), (0.62, 0.58), (0.7, 0.55)]:
        print(f"{name}, {p_a}, {p_b}:   ")
        check(lambda: sim_match(p_a, p_b, sim_game), match_prob(p_a, p_b, hold))

Golden, 0.6, 0.6:   
Sim: 0.4981   Exact: 0.5000   SE: 0.0035   OK
Golden, 0.62, 0.58:   
Sim: 0.6815   Exact: 0.6805   SE: 0.0033   OK
Golden, 0.7, 0.55:   
Sim: 0.9603   Exact: 0.9592   SE: 0.0014   OK
Deuce, 0.6, 0.6:   
Sim: 0.4986   Exact: 0.5000   SE: 0.0035   OK
Deuce, 0.62, 0.58:   
Sim: 0.7019   Exact: 0.6974   SE: 0.0032   OK
Deuce, 0.7, 0.55:   
Sim: 0.9692   Exact: 0.9705   SE: 0.0012   OK
Star, 0.6, 0.6:   
Sim: 0.5037   Exact: 0.5000   SE: 0.0035   OK
Star, 0.62, 0.58:   
Sim: 0.6953   Exact: 0.6942   SE: 0.0033   OK
Star, 0.7, 0.55:   
Sim: 0.9680   Exact: 0.9689   SE: 0.0012   OK


In [348]:
check(lambda: sim_match(0.6, 0.6), 0.5)                       # equal teams: 50/50
check(lambda: sim_match(0.62, 0.58, server="B"),
      match_prob(0.62, 0.58))                                 # B serving first: same answer
print(sim_match(1.0, 0.0))

Sim: 0.4979   Exact: 0.5000   SE: 0.0035   OK
Sim: 0.6923   Exact: 0.6942   SE: 0.0033   OK
True
